# DomainNet-126 **eta-sweep** — TFF on resnet50 (real→clipart, seed 42)

Decisive single-target slice (arch=resnet50, source=real, target=clipart, seed=42 ONLY).
The method is `heat` in code; **results are labeled TFF**. ~8–12 h on L4; descending-eta
order lands the informative high-eta points first, so a partial plot is useful early.

It answers THREE questions (reported separately, not one pass/fail):
- **Q1 (LAW):** does p\* scale linearly with η·‖ḡ‖ on ResNet-50/DomainNet (a per-arch R)?
- **Q2 (USEFULNESS):** does TFF's best-over-p accuracy beat **BN-adapt** at any η? (Branch 1
  "TFF wins" vs Branch 2 "graceful fallback").
- **Q3 (MECHANISM):** are collapse boundaries **drift-collapse** (params diverge — the law's
  mechanism) or **signal-collapse** (accuracy craters to chance, drift bounded — a different
  axis)?

Boundary = HARD collapse only (nan/inf OR mean_acc ≤ ~0.02), since `soft:below_source` fires
at every p here. BN stays in **train mode** (the smoke test proved frozen → NaN). Resumable.

## 1. GPU check

In [ ]:
import subprocess
o = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(o.stdout if o.returncode==0 else "⚠️ No GPU — set Runtime → GPU (L4/A100). ~8–12h on L4.")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = "https://github.com/octadion/heat.git"
REPO_DIR   = "heat"
GIT_BRANCH = "formulation"      # branch holding the DomainNet code (push there first!)
USE_DRIVE  = True
# eta-sweep results (Drive-backed, resumable). Keep SEPARATE from CIFAR results.
DRIVE_DIR  = "/content/drive/MyDrive/pstar_domainnet_clipart"
DOMAINNET_ROOT = "/content/domainnet-126"
DOWNLOAD_REAL  = False          # slice reads ONLY clipart images + the real checkpoint

TARGET   = "clipart"
SEED     = 42
# eta grid DESCENDING (high-signal points first). Per-run eta on the x-axis.
ETAS     = [1.6e-2, 8e-3, 4e-3, 2e-3, 1e-3, 5e-4, 2e-4]
P_GRID   = [0.0, 0.005, 0.010, 0.020]   # BASE coarse (scaled by eta/1e-3); +{0.1,0.2} auto
BISECT_STEPS = 3
CHANCE_ACC   = 0.02             # 126-class chance ~0.008; hard-collapse floor
BATCH_SIZE, NUM_WORKERS = 64, 2

CKPT_DRIVE_FOLDER = "https://drive.google.com/drive/folders/16vTNNzzAt4M1mmeLsOxSFDRzBogaNkJw"
LIST_URL = "https://raw.githubusercontent.com/DianCh/AdaContrast/master/datasets/domainnet-126/{domain}_list.txt"
IMG_URL  = {"clipart": "http://csr.bu.edu/ftp/visda/2019/multi-source/groundtruth/clipart.zip",
            "real":    "http://csr.bu.edu/ftp/visda/2019/multi-source/real.zip"}
# =======================================================================
RESULTS_DIR = DRIVE_DIR if USE_DRIVE else "/content/pstar_domainnet_clipart"
print("RESULTS_DIR =", RESULTS_DIR, "| TARGET =", TARGET)

## 3. Mount Drive + clone/refresh repo (branch `formulation`)

In [ ]:
import os, subprocess
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone"] + (["-b",GIT_BRANCH] if GIT_BRANCH else []) + [REPO_URL,REPO_DIR], check=True)
else:
    if GIT_BRANCH: subprocess.run(["git","-C",REPO_DIR,"checkout",GIT_BRANCH], check=False)
    subprocess.run(["git","-C",REPO_DIR,"pull"], check=False)
os.chdir("/content/"+REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
subprocess.run(["pip","install","-q","-r","requirements.txt"], check=False)
subprocess.run(["pip","install","-q","gdown"], check=False)
os.environ["PYTHONPATH"] = os.getcwd()+os.pathsep+os.environ.get("PYTHONPATH","")
print("branch:", subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"],capture_output=True,text=True).stdout.strip())

## 4. Get clipart list + images (real images NOT needed for this slice)

In [ ]:
import os, subprocess
os.makedirs(DOMAINNET_ROOT, exist_ok=True)
for domain in (["clipart","real"] if DOWNLOAD_REAL else ["clipart"]):
    dst = os.path.join(DOMAINNET_ROOT, f"{domain}_list.txt")
    if not os.path.exists(dst):
        subprocess.run(["wget","-q","-O",dst, LIST_URL.format(domain=domain)], check=True)
    print("[list]", dst, "lines:", sum(1 for _ in open(dst)))
    if not os.path.isdir(os.path.join(DOMAINNET_ROOT, domain)):
        zp = os.path.join(DOMAINNET_ROOT, f"{domain}.zip")
        print(f"[wget] {IMG_URL[domain]}"); subprocess.run(["wget","-q","-O",zp, IMG_URL[domain]], check=True)
        print(f"[unzip] {zp}"); subprocess.run(["unzip","-q",zp,"-d",DOMAINNET_ROOT], check=True); os.remove(zp)
print("[ok] data under", DOMAINNET_ROOT)

## 5. Get the source=real checkpoint (AdaContrast `best_real_2020.pth.tar`)

In [ ]:
import os, glob, subprocess
CKPT_DIR = "experiments/checkpoints/domainnet126_source"
os.makedirs(CKPT_DIR, exist_ok=True)
def find_real_ckpt():
    exts=(".pth.tar",".pth",".pt",".ckpt",".tar")
    for pat in ("*real*2020*","*real*"):
        hits=[f for f in glob.glob(os.path.join(CKPT_DIR,"**",pat),recursive=True) if f.lower().endswith(exts)]
        if hits: return sorted(hits)
    return []
found = find_real_ckpt()
if not found:
    subprocess.run(["gdown","--folder","--remaining-ok",CKPT_DRIVE_FOLDER,"-O",CKPT_DIR], check=True)
    found = find_real_ckpt()
if not found:
    print("[diag] files under", CKPT_DIR, ":")
    for f in sorted(glob.glob(os.path.join(CKPT_DIR,"**","*"),recursive=True)):
        if os.path.isfile(f): print(f"    {f} ({os.path.getsize(f)//1024//1024} MB)")
    raise AssertionError("Set CKPT_REAL to the real-source file above.")
CKPT_REAL = found[0]; print("CKPT_REAL =", CKPT_REAL)

## 6. Run the eta-sweep (resumable, ~8–12 h L4)
Baselines (source, bn_adapt) once, then heat over the eta-scaled p-grid + {0.1,0.2} at each
η (descending), bisecting the HARD-collapse boundary. **Re-run after any disconnect** — it
skips eta-tagged JSONs already on Drive.

In [ ]:
import subprocess
cmd = [
    "python", "scripts/run_domainnet_eta_sweep.py",
    "--results-dir", RESULTS_DIR, "--data-root", DOMAINNET_ROOT,
    "--ckpt-real", CKPT_REAL, "--target-domain", TARGET,
    "--etas", *[repr(e) for e in ETAS],
    "--p-grid", *[repr(p) for p in P_GRID],
    "--bisect-steps", str(BISECT_STEPS), "--chance-acc", repr(CHANCE_ACC),
    "--seed", str(SEED), "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
]
print("$", " ".join(cmd), "\n"); subprocess.run(cmd, check=True)

## 7. Analyze — Q1 (law) / Q2 (usefulness vs BN-adapt) / Q3 (mechanism)
Displays the law plot (mechanism-colored) + the usefulness table + the 3-line verdict.

In [ ]:
import subprocess, os
from IPython.display import Image, Markdown, display
subprocess.run([
    "python", "scripts/analyze_domainnet_clipart.py",
    "--results-dir", RESULTS_DIR, "--seed", str(SEED), "--chance-acc", repr(CHANCE_ACC),
], check=True)
png = os.path.join(RESULTS_DIR, "analysis", "domainnet_clipart_law.png")
md_ = os.path.join(RESULTS_DIR, "analysis", "domainnet_clipart_usefulness.md")
if os.path.exists(png): display(Image(filename=png))
if os.path.exists(md_): display(Markdown(open(md_, encoding="utf-8").read()))